# SQS суперячейки ХСИС Al-Co-Cr-Fe-Ni-Ti (L12)
Упрочняющая фаза **(Ni,Co,Fe,Cr)₃(Al,Ti)** со структурой **L12** (Pm-3m, #221)

Генерация **Special Quasirandom Structure (SQS)** с помощью библиотеки **sqsgenerator**.
Режим `sublattice_mode: split` — оптимизация A и B подрешёток раздельно.

In [10]:
%pip install sqsgenerator pymatgen

In [11]:
from sqsgenerator import parse_config, optimize, write
from pymatgen.core import Structure

## 1. Параметры

In [12]:
a = 3.57
n = 2  # 2x2x2 -> 32 атома

## 2. Конфигурация sqsgenerator

In [13]:
coords = []
species = []

for ix in range(n):
    for iy in range(n):
        for iz in range(n):
            coords.append([ix/n, iy/n, iz/n])
            species.append("Al")
            for offset in [[0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]]:
                coords.append([(ix+offset[0])/n, (iy+offset[1])/n, (iz+offset[2])/n])
                species.append("Ni")

config = {
    "iterations": 10000000,
    "sublattice_mode": "split",
    "structure": {
        "lattice": [[a*n, 0, 0], [0, a*n, 0], [0, 0, a*n]],
        "coords": coords,
        "species": species,
        "supercell": [1, 1, 1]
    },
    "composition": [
        {"sites": "Al", "Al": 6, "Ti": 2},
        {"sites": "Ni", "Ni": 12, "Co": 5, "Fe": 4, "Cr": 3}
    ]
}

print(f"Сайтов: {len(coords)}")

Сайтов: 32


## 3. Запуск SQS оптимизации

In [14]:
cfg = parse_config(config)
print("Запуск sqsgenerator (C++, работает быстро)...")
pack = optimize(cfg)
print(f"Найдено решений: {len(pack)}")
for i, (obj, sols) in enumerate(pack):
    print(f"  Objective #{i}: {obj:.6f} ({len(sols)} структур)")

Запуск sqsgenerator (C++, работает быстро)...
Найдено решений: 33
  Objective #0: 5.413889 (1 структур)
  Objective #1: 5.658333 (1 структур)
  Objective #2: 5.709722 (1 структур)
  Objective #3: 5.747222 (7 структур)
  Objective #4: 5.784722 (1 структур)
  Objective #5: 5.866667 (1 структур)
  Objective #6: 5.884722 (1 структур)
  Objective #7: 6.018056 (1 структур)
  Objective #8: 6.047222 (1 структур)
  Objective #9: 6.329167 (1 структур)
  Objective #10: 6.447222 (1 структур)
  Objective #11: 6.572222 (1 структур)
  Objective #12: 6.747222 (1 структур)
  Objective #13: 6.787500 (1 структур)
  Objective #14: 6.925000 (1 структур)
  Objective #15: 7.145833 (1 структур)
  Objective #16: 7.225000 (1 структур)
  Objective #17: 7.698611 (1 структур)
  Objective #18: 8.334722 (1 структур)
  Objective #19: 8.397222 (1 структур)
  Objective #20: 8.501389 (1 структур)
  Objective #21: 8.725000 (1 структур)
  Objective #22: 8.858333 (1 структур)
  Objective #23: 9.005556 (1 структур)
  Object

## 4. Экспорт лучшего SQS

In [15]:
best = pack.best()
print(f"Лучший objective: {best.objective:.6f}")

# Экспорт через sqsgenerator (формат определяется расширением файла)
write(best.structure(), "HEA_L12_SQS_32atoms.cif")
print("Сохранен: HEA_L12_SQS_32atoms.cif")

Лучший objective: 5.413889
Сохранен: HEA_L12_SQS_32atoms.cif


In [16]:
!sed -i 's/0+//g' HEA_L12_SQS_32atoms.cif

## 5. Проверка через pymatgen

In [17]:
sqs_structure = Structure.from_file("HEA_L12_SQS_32atoms.cif")

print(f"SQS формула: {sqs_structure.composition.formula}")
print(f"Число атомов: {sqs_structure.num_sites}")
print(f"Параметры решетки: {sqs_structure.lattice.a:.4f} x {sqs_structure.lattice.b:.4f} x {sqs_structure.lattice.c:.4f}")
print(f"\nСостав по элементам:")
for el, amt in sqs_structure.composition.as_dict().items():
    print(f"  {el}: {amt:.0f} ({amt/sqs_structure.num_sites*100:.1f}%)")

SQS формула: Ti2 Al6 Cr3 Fe4 Co5 Ni12
Число атомов: 32
Параметры решетки: 7.1400 x 7.1400 x 7.1400

Состав по элементам:
  Ti0+: 2 (6.2%)
  Al0+: 6 (18.8%)
  Cr0+: 3 (9.4%)
  Fe0+: 4 (12.5%)
  Co0+: 5 (15.6%)
  Ni0+: 12 (37.5%)


In [18]:
print("Позиции атомов (дробные координаты):")
print(f"{'#':>3}  {'Element':>7}  {'x':>10}  {'y':>10}  {'z':>10}")
print("-" * 45)
for i, site in enumerate(sqs_structure):
    fc = site.frac_coords
    print(f"{i+1:>3}  {site.species_string:>7}  {fc[0]:>10.4f}  {fc[1]:>10.4f}  {fc[2]:>10.4f}")

Позиции атомов (дробные координаты):
  #  Element           x           y           z
---------------------------------------------
  1     Ti0+      0.5000      0.0000      0.5000
  2     Ti0+      0.5000      0.0000      0.0000
  3     Al0+      0.0000      0.0000      0.0000
  4     Al0+      0.0000      0.5000      0.5000
  5     Al0+      0.0000      0.5000      0.0000
  6     Al0+      0.5000      0.5000      0.0000
  7     Al0+      0.0000      0.0000      0.5000
  8     Al0+      0.5000      0.5000      0.5000
  9     Cr0+      0.5000      0.2500      0.7500
 10     Cr0+      0.2500      0.7500      0.0000
 11     Cr0+      0.7500      0.0000      0.7500
 12     Fe0+      0.7500      0.5000      0.7500
 13     Fe0+      0.0000      0.7500      0.2500
 14     Fe0+      0.7500      0.0000      0.2500
 15     Fe0+      0.0000      0.7500      0.7500
 16     Co0+      0.7500      0.7500      0.0000
 17     Co0+      0.2500      0.2500      0.0000
 18     Co0+      0.2500      0.500